### Part-A

- 1. What is Regularization in Machine Learning? Why is it needed?

    Regularization is a technique used to reduce overfitting in a machine learning model.
    It adds a penalty to large model coefficients. This helps the model perform better on new and unseen data.

- 2. Difference between Ridge Regression (L2) and Lasso Regression (L1)

    Ridge Regression uses L2 regularization, which reduces the size of coefficients but usually does not make them zero.
    Lasso Regression uses L1 regularization, which can make some coefficients exactly zero. Therefore, Lasso can also help in selecting important features.

- 3. What is Cross-Validation and why is it important?

    Cross-validation is a technique used to check how well a model performs on different parts of the dataset.
    It helps us select a better model and reduces the chance of depending only on one train-test split.

- 4. Explain the following cross-validation techniques

    K-Fold Cross-Validation:
    The dataset is divided into K parts. The model is trained on K-1 parts and tested on the remaining part. This is repeated K times.

    Stratified K-Fold Cross-Validation:
    It keeps similar proportions of the target values in each fold. It is mainly useful for classification problems.

    Leave-One-Out Cross-Validation (LOOCV):
    One data point is used for testing and all other points are used for training. This is repeated for every data point.

    Time Series Split:
    The data is divided according to time. The model uses earlier data for training and later data for testing.

- 5. Why are tree-based models less sensitive to feature scaling?

    Tree-based models split data using feature values.
    They do not depend on the distance between values, so scaling features is generally not required for Decision Tree and Random Forest models.

Import statments

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneOut, TimeSeriesSplit, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.svm import SVR

### Part B: Dataset Understanding and Prepraration

In [4]:
df = pd.read_csv("D:\\RW\\RW_Exam\\ML\\PR-2\\Advanced_Regression_HousePrice_Dataset_3800 - Advanced_Regression_HousePrice_Dataset_3800.csv.csv")
display(df.head())

,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [5]:
target = 'house_price_inr'
X = df.drop(columns=[target, 'property_id'])
y = df[target]

print(df.shape)
print(df.info())
print(df.isnull().sum())


(3800, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3800 entries, 0 to 3799
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   property_id       3800 non-null   int64  
 1   sale_date         3800 non-null   object 
 2   area_sqft         3800 non-null   int64  
 3   bedrooms          3800 non-null   int64  
 4   bathrooms         3800 non-null   int64  
 5   location_score    3800 non-null   float64
 6   property_age      3800 non-null   int64  
 7   distance_city_km  3800 non-null   float64
 8   near_school       3800 non-null   int64  
 9   near_metro        3800 non-null   int64  
 10  crime_rate_index  3800 non-null   float64
 11  house_price_inr   3800 non-null   int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 356.4+ KB
None
property_id         0
sale_date           0
area_sqft           0
bedrooms            0
bathrooms           0
location_score      0
property_age        0
d

In [6]:
df['sale_date'] = pd.to_datetime(df['sale_date'])

df['sale_year'] = df['sale_date'].dt.year
df['sale_month'] = df['sale_date'].dt.month
df['sale_day'] = df['sale_date'].dt.day

df = df.drop(columns=['sale_date'])

X = df.drop(columns=['house_price_inr', 'property_id'])
y = df['house_price_inr']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print(X_train.shape)
print(X_test.shape)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

(3040, 12)
(760, 12)


### Part C: Regularized Linear Models

In [12]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)

lasso = Lasso(alpha=1.0)
lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)

In [13]:
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(),ridge_params,cv=5,scoring='neg_mean_squared_error')
ridge_grid.fit(X_train_scaled, y_train)

print("Best Ridge Alpha:", ridge_grid.best_params_)

lasso_params = {'alpha': [0.01, 0.1, 1, 10, 100]}
lasso_grid = GridSearchCV(Lasso(),lasso_params,cv=5,scoring='neg_mean_squared_error')
lasso_grid.fit(X_train_scaled, y_train)

print("Best Lasso Alpha:", lasso_grid.best_params_)

Best Ridge Alpha: {'alpha': 1}
Best Lasso Alpha: {'alpha': 100}


In [14]:
ridge_train_pred = best_ridge.predict(X_train_scaled)
ridge_test_pred = best_ridge.predict(X_test_scaled)

lasso_train_pred = best_lasso.predict(X_train_scaled)
lasso_test_pred = best_lasso.predict(X_test_scaled)

ridge_train_error = mean_squared_error(y_train, ridge_train_pred)
ridge_test_error = mean_squared_error(y_test, ridge_test_pred)

lasso_train_error = mean_squared_error(y_train, lasso_train_pred)
lasso_test_error = mean_squared_error(y_test, lasso_test_pred)

print("Ridge Training Error:", ridge_train_error)
print("Ridge Validation Error:", ridge_test_error)

print("Lasso Training Error:", lasso_train_error)
print("Lasso Validation Error:", lasso_test_error)

coefficient_comparison = pd.DataFrame({
    'Feature': X.columns,
    'Ridge Coefficient': best_ridge.coef_,
    'Lasso Coefficient': best_lasso.coef_
})

display(coefficient_comparison)
comparison = pd.DataFrame({
    'Model': ['Ridge', 'Lasso'],
    'Training Error': [ridge_train_error, lasso_train_error],
    'Validation Error': [ridge_test_error, lasso_test_error]
})

display(comparison)


NameError: name 'best_ridge' is not defined

### Part D: Cross-Validation Strategies

In [15]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

kfold_scores = cross_val_score(model,X,y,cv=kfold,scoring='neg_mean_squared_error')
kfold_rmse = np.sqrt(-kfold_scores)

print("K-Fold RMSE:", kfold_rmse)
print("Mean K-Fold RMSE:", kfold_rmse.mean())

K-Fold RMSE: [2539952.39652634 2425201.22968229 2654132.04031752 2405992.76726241
 2472550.43486967]
Mean K-Fold RMSE: 2499565.7737316475


In [16]:
y_bins = pd.qcut(y, q=5, labels=False)

stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model,X,y,cv=stratified.split(X, y_bins),scoring='neg_mean_squared_error')

stratified_rmse = np.sqrt(-scores)

print("Stratified K-Fold RMSE:", stratified_rmse)
print("Mean RMSE:", stratified_rmse.mean())

Stratified K-Fold RMSE: [2478380.30734097 2418138.38045502 2482313.36033471 2526612.84994395
 2591829.50071463]
Mean RMSE: 2499454.879757855


In [17]:
loo = LeaveOneOut()

loo_scores = cross_val_score(model,X,y,cv=loo,scoring='neg_mean_squared_error')
loo_rmse = np.sqrt(-loo_scores)

print("LOOCV Mean RMSE:", loo_rmse.mean())

LOOCV Mean RMSE: 1912757.0650831505


In [18]:
time_data = df.sort_values('sale_year')

X_time = time_data.drop(columns=['house_price_inr', 'property_id'])
y_time = time_data['house_price_inr']

time_split = TimeSeriesSplit(n_splits=5)

time_scores = cross_val_score(
    model,
    X_time,
    y_time,
    cv=time_split,
    scoring='neg_mean_squared_error'
)

time_rmse = np.sqrt(-time_scores)

print("Time Series RMSE:", time_rmse)
print("Mean Time Series RMSE:", time_rmse.mean())

Time Series RMSE: [2594521.6859807  2474096.61368    2372765.77261383 2524468.50567955
 2508156.53156036]
Mean Time Series RMSE: 2494801.821902887


In [19]:
cv_results = pd.DataFrame({
    'CV Method': [
        'K-Fold',
        'Stratified K-Fold',
        'LOOCV',
        'Time Series Split'
    ],
    'Mean RMSE': [
        kfold_rmse.mean(),
        stratified_rmse.mean(),
        loo_rmse.mean(),
        time_rmse.mean()
    ]
})

cv_results

,CV Method,Mean RMSE
0,K-Fold,2.499566e+06
1,Stratified K-Fold,2.499455e+06
2,LOOCV,1.912757e+06
3,Time Series Split,2.494802e+06


### Part E: Tree-Based Regression Models

In [20]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

tree = DecisionTreeRegressor(max_depth=5,min_samples_split=10,random_state=42)
tree.fit(X_train, y_train)

tree_pred = tree.predict(X_test)

In [21]:
forest = RandomForestRegressor(n_estimators=100,random_state=42)
forest.fit(X_train, y_train)

forest_pred = forest.predict(X_test)

In [22]:
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
tree_mae = mean_absolute_error(y_test, tree_pred)
tree_r2 = r2_score(y_test, tree_pred)

forest_rmse = np.sqrt(mean_squared_error(y_test, forest_pred))
forest_mae = mean_absolute_error(y_test, forest_pred)
forest_r2 = r2_score(y_test, forest_pred)

print("Decision Tree")
print("RMSE:", tree_rmse)
print("MAE:", tree_mae)
print("R2 Score:", tree_r2)

print("\nRandom Forest")
print("RMSE:", forest_rmse)
print("MAE:", forest_mae)
print("R2 Score:", forest_r2)

Decision Tree
RMSE: 3063631.2537640957
MAE: 2325303.8928159946
R2 Score: 0.8834566744111698

Random Forest
RMSE: 2406259.2553653875
MAE: 1758820.2308026315
R2 Score: 0.9281049055567916


In [23]:
tree_forest_comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'RMSE': [tree_rmse, forest_rmse],
    'MAE': [tree_mae, forest_mae],
    'R2 Score': [tree_r2, forest_r2]
})

tree_forest_comparison

,Model,RMSE,MAE,R2 Score
0,Decision Tree,3.063631e+06,2.325304e+06,0.883457
1,Random Forest,2.406259e+06,1.758820e+06,0.928105


### Part F: Support Vector Regression (SVR)

In [24]:
svr_linear = SVR(kernel='linear')
svr_linear.fit(X_train_scaled, y_train)

linear_pred = svr_linear.predict(X_test_scaled)

svr_poly = SVR(kernel='poly', degree=2)
svr_poly.fit(X_train_scaled, y_train)

poly_pred = svr_poly.predict(X_test_scaled)

svr_rbf = SVR(kernel='rbf')
svr_rbf.fit(X_train_scaled, y_train)

rbf_pred = svr_rbf.predict(X_test_scaled)

In [25]:
svr_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.1, 1],
    'epsilon': [0.1, 0.2]
}

svr_grid = GridSearchCV(
    SVR(kernel='rbf'),
    svr_params,
    cv=5,
    scoring='neg_mean_squared_error'
)

svr_grid.fit(X_train_scaled, y_train)

print("Best Parameters:", svr_grid.best_params_)

best_svr = svr_grid.best_estimator_

svr_pred = best_svr.predict(X_test_scaled)

Best Parameters: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale'}


In [26]:
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_pred))
rbf_rmse = np.sqrt(mean_squared_error(y_test, rbf_pred))
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
forest_rmse = np.sqrt(mean_squared_error(y_test, forest_pred))

svr_comparison = pd.DataFrame({
    'Model': [
        'SVR Linear',
        'SVR Polynomial',
        'SVR RBF',
        'Decision Tree',
        'Random Forest'
    ],
    'RMSE': [
        linear_rmse,
        poly_rmse,
        rbf_rmse,
        tree_rmse,
        forest_rmse
    ]
})

svr_comparison

,Model,RMSE
0,SVR Linear,8.982717e+06
1,SVR Polynomial,8.987447e+06
2,SVR RBF,8.987351e+06
3,Decision Tree,3.063631e+06
4,Random Forest,2.406259e+06


### Part G: Model Comparison and Evaluation 

In [27]:
def evaluate_model(name, y_test, prediction):
    mse = mean_squared_error(y_test, prediction)
    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, prediction)

    return [name, mse, mae, rmse, r2]

results = []

results.append(evaluate_model('Ridge', y_test, ridge_pred))
results.append(evaluate_model('Lasso', y_test, lasso_pred))
results.append(evaluate_model('Decision Tree', y_test, tree_pred))
results.append(evaluate_model('Random Forest', y_test, forest_pred))
results.append(evaluate_model('SVR Linear', y_test, linear_pred))
results.append(evaluate_model('SVR Polynomial', y_test, poly_pred))
results.append(evaluate_model('SVR RBF', y_test, rbf_pred))

results_df = pd.DataFrame(
    results,
    columns=['Model', 'MSE', 'MAE', 'RMSE', 'R2 Score']
)

results_df

,Model,MSE,MAE,RMSE,R2 Score
0,Ridge,6.451358e+12,1.945385e+06,2.539952e+06,0.919894
1,Lasso,6.450122e+12,1.945520e+06,2.539709e+06,0.919909
2,Decision Tree,9.385836e+12,2.325304e+06,3.063631e+06,0.883457
3,Random Forest,5.790084e+12,1.758820e+06,2.406259e+06,0.928105
4,SVR Linear,8.068921e+13,6.990274e+06,8.982717e+06,-0.001913
5,SVR Polynomial,8.077421e+13,6.994137e+06,8.987447e+06,-0.002968
6,SVR RBF,8.077247e+13,6.994047e+06,8.987351e+06,-0.002947


In [31]:
models = [
    Ridge(alpha=1.0),
    Lasso(alpha=1.0),
    DecisionTreeRegressor(max_depth=5, min_samples_split=10, random_state=42),
    RandomForestRegressor(n_estimators=100, random_state=42),
    SVR(kernel='rbf')]

names = [
    'Ridge',
    'Lasso',
    'Decision Tree',
    'Random Forest',
    'SVR RBF'
]

train_errors = []
test_errors = []

for model1, name in zip(models, names):
    if name in ['Ridge', 'Lasso', 'SVR RBF']:
        model1.fit(X_train_scaled, y_train)
        train_pred = model1.predict(X_train_scaled)
        test_pred = model1.predict(X_test_scaled)
    else:
        model1.fit(X_train, y_train)
        train_pred = model1.predict(X_train)
        test_pred = model1.predict(X_test)

    train_errors.append(np.sqrt(mean_squared_error(y_train, train_pred)))
    test_errors.append(np.sqrt(mean_squared_error(y_test, test_pred)))

In [32]:
status = []
for train, test in zip(train_errors, test_errors):
    if train > test and train > test * 0.9:
        status.append("Underfitting")
    elif test > train * 1.5:
        status.append("Overfitting")
    else:
        status.append("Good Fit")

error_comparison = pd.DataFrame({
    'Model': names,
    'Training RMSE': train_errors,
    'Testing RMSE': test_errors,
    'Status': status
})

error_comparison

,Model,Training RMSE,Testing RMSE,Status
0,Ridge,2.482399e+06,2.539952e+06,Good Fit
1,Lasso,2.482395e+06,2.539709e+06,Good Fit
2,Decision Tree,2.739161e+06,3.063631e+06,Good Fit
3,Random Forest,8.941688e+05,2.406259e+06,Overfitting
4,SVR RBF,8.667162e+06,8.987351e+06,Good Fit


### Part F:Final Analysis and Reporting

In [34]:
results_df.sort_values('RMSE')

,Model,MSE,MAE,RMSE,R2 Score
3,Random Forest,5.790084e+12,1.758820e+06,2.406259e+06,0.928105
1,Lasso,6.450122e+12,1.945520e+06,2.539709e+06,0.919909
0,Ridge,6.451358e+12,1.945385e+06,2.539952e+06,0.919894
2,Decision Tree,9.385836e+12,2.325304e+06,3.063631e+06,0.883457
4,SVR Linear,8.068921e+13,6.990274e+06,8.982717e+06,-0.001913
6,SVR RBF,8.077247e+13,6.994047e+06,8.987351e+06,-0.002947
5,SVR Polynomial,8.077421e+13,6.994137e+06,8.987447e+06,-0.002968


In [35]:
best_model = results_df.loc[results_df['RMSE'].idxmin(), 'Model']

print("Model with lowest RMSE:", best_model)

Model with lowest RMSE: Random Forest


# Final Report

#### Project Overview

This project focused on predicting house prices using different regression techniques. The dataset contains information about properties such as area, bedrooms, bathrooms, location score, property age, distance from city, nearby school, nearby metro and crime rate.

The target variable used for prediction was `house_price_inr`.

#### Data Preparation

The dataset was checked for missing values and unnecessary columns. `property_id` was removed because it was only an identifier.

The `sale_date` column was converted into date format and useful date features were extracted.

The dataset was divided into **80% training data and 20% testing data**.

`StandardScaler` was used for models such as Ridge, Lasso and SVR.

#### Models Used

The following models were implemented:

1. Ridge Regression
2. Lasso Regression
3. Decision Tree Regression
4. Random Forest Regression
5. SVR with Linear Kernel
6. SVR with Polynomial Kernel
7. SVR with RBF Kernel

#### Cross-Validation Techniques

The following cross-validation techniques were studied:

- K-Fold Cross-Validation
- Stratified K-Fold Cross-Validation
- Leave-One-Out Cross-Validation
- Time Series Split

#### Model Evaluation

The models were evaluated using:

- MSE
- MAE
- RMSE
- R² Score

RMSE was mainly used to compare the prediction errors. A lower RMSE means lower prediction error.

#### Final Model

Based on the test results, **Random Forest Regression gave the lowest RMSE among the tested models**.

Random Forest uses multiple decision trees and combines their predictions. It can capture non-linear relationships between property features and house prices.

#### Overfitting and Underfitting

Training and testing RMSE were compared to check the model behavior.

A large difference between training and testing error can indicate overfitting, while high errors on both datasets can indicate underfitting.

#### Conclusion

Different regression models were tested on the house price dataset.

Ridge and Lasso helped control model coefficients, while Decision Tree and Random Forest were able to capture non-linear relationships. SVR was also tested using different kernels.

After comparing the models using MSE, MAE, RMSE and R², **Random Forest Regression was the final model in this experiment because it produced the lowest test RMSE**.

The model can be used to predict house prices based on the available property features.